# Chapter 11 · Active Learning and Bayesian Optimisation — live notebook

This notebook runs entirely in your browser via the Pyodide kernel.
All state is saved to your browser's local storage; nothing is sent to a server.

Back to the chapter: <https://dongzhaohe321418-lab.github.io/materials-simulation-handbook/ch11-active/>

A Gaussian process regressor with an RBF kernel, and a Bayesian optimisation loop on a toy 1D objective using Expected Improvement.

Only Pyodide-compatible packages are used (numpy, scipy, matplotlib, ipywidgets).


## 11.1 Gaussian process regression on noisy samples


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(3)

def rbf(x1, x2, length=0.4, sigma=1.0):
    d2 = (x1[:, None] - x2[None, :]) ** 2
    return sigma ** 2 * np.exp(-0.5 * d2 / length ** 2)

def gp_predict(x_train, y_train, x_test, noise=0.05, length=0.4):
    K = rbf(x_train, x_train, length) + noise ** 2 * np.eye(len(x_train))
    Ks = rbf(x_train, x_test, length)
    Kss = rbf(x_test, x_test, length)
    L = np.linalg.cholesky(K)
    alpha = np.linalg.solve(L.T, np.linalg.solve(L, y_train))
    mean = Ks.T @ alpha
    v = np.linalg.solve(L, Ks)
    var = np.diag(Kss) - np.sum(v ** 2, axis=0)
    return mean, np.sqrt(np.clip(var, 0, None))

def true_f(x):
    return np.sin(3 * x) + 0.3 * x

x_train = rng.uniform(-2, 2, 8)
y_train = true_f(x_train) + rng.normal(0, 0.1, len(x_train))
x_test = np.linspace(-2.5, 2.5, 300)
mu, std = gp_predict(x_train, y_train, x_test)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x_test, true_f(x_test), 'k--', lw=1, label='truth')
ax.plot(x_train, y_train, 'ko', label='observations')
ax.plot(x_test, mu, 'C0', label='GP mean')
ax.fill_between(x_test, mu - 2 * std, mu + 2 * std, alpha=0.2, color='C0', label='+/- 2 sigma')
ax.legend(fontsize=8)
ax.set_title('Gaussian process regression')
plt.show()


## 11.2 Bayesian optimisation with Expected Improvement

Maximise a 1D objective by alternating GP fits and EI-driven queries. Watch the algorithm zero in on the optimum within a handful of evaluations.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import erf, sqrt

def phi(z):
    return np.exp(-0.5 * z ** 2) / np.sqrt(2 * np.pi)

def Phi(z):
    return 0.5 * (1 + np.vectorize(lambda x: erf(x / sqrt(2)))(z))

def ei(mu, std, best, xi=0.01):
    std = np.clip(std, 1e-9, None)
    z = (mu - best - xi) / std
    return (mu - best - xi) * Phi(z) + std * phi(z)

def objective(x):
    return -((x - 0.4) ** 2) + 0.3 * np.sin(8 * x)

rng = np.random.default_rng(0)
x_grid = np.linspace(-1, 1, 400)
x_obs = rng.uniform(-1, 1, 3)
y_obs = objective(x_obs)

fig, axes = plt.subplots(2, 3, figsize=(11, 5), sharex=True)
for step, ax in enumerate(axes.ravel()):
    mu, std = gp_predict(x_obs, y_obs, x_grid, noise=0.01, length=0.15)
    acq = ei(mu, std, y_obs.max())
    x_next = x_grid[int(np.argmax(acq))]
    ax.plot(x_grid, objective(x_grid), 'k--', lw=0.8, label='truth')
    ax.plot(x_grid, mu, 'C0')
    ax.fill_between(x_grid, mu - 2 * std, mu + 2 * std, alpha=0.15, color='C0')
    ax.plot(x_obs, y_obs, 'ko')
    ax.axvline(x_next, color='C3', lw=0.8)
    ax.set_title(f'after {step} suggestions')
    x_obs = np.append(x_obs, x_next)
    y_obs = np.append(y_obs, objective(x_next))
fig.suptitle('Bayesian optimisation under Expected Improvement')
fig.tight_layout()
plt.show()
print(f'best objective found: {y_obs.max():.4f} at x = {x_obs[int(np.argmax(y_obs))]:.4f}')
